# 04 – Explainability

Nesta etapa, nosso grupo busca interpretar as decisões do modelo e entender o papel de cada variável nas predições. Utilizamos quatro técnicas complementares:

1. **SHAP** – _Quais variáveis mais influenciaram esta predição?_
2. **LIME** – _Como pequenas mudanças nas variáveis afetariam esta predição?_
3. **Permutation Importance** – _O que acontece com a performance do modelo se embaralharmos esta variável?_
4. **Counterfactual Explanations** – _O que precisaria mudar para que o modelo tomasse uma decisão diferente?_

In [1]:
import pandas as pd
import numpy as np
import shap
import lime.lime_tabular
import matplotlib.pyplot as plt
import joblib
from sklearn.inspection import permutation_importance

ModuleNotFoundError: No module named 'shap'

SHAP

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

LIME

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_test),
    feature_names=X_test.columns.tolist(),
    class_names=['Eficiente', 'Ineficiente'],
    mode='classification'
)

i = 0
exp = explainer.explain_instance(X_test.iloc[i], model.predict_proba)
exp.show_in_notebook()

PERMUTATION IMPORTANCE

In [ ]:
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
}).sort_values(by="importance", ascending=False)

importance_df.plot(kind="bar", x="feature", y="importance", legend=False)
plt.title("Permutation Importance")
plt.tight_layout()
plt.show()

COUNTERFACTUAL EXPLANATIONS

In [ ]:
# pip install dice-ml 
import dice_ml
from dice_ml.utils import helpers

# Define dados e modelo para contrafactuais
data_dice = dice_ml.Data(dataframe=X_test.assign(label_ineficiente=y_test),
                         continuous_features=X_test.columns.tolist(),
                         outcome_name='label_ineficiente')

model_dice = dice_ml.Model(model=model, backend='sklearn')
exp = dice_ml.Dice(data_dice, model_dice)

#Gera contrafactuais para o primeiro exemplo
query_instance = X_test.iloc[[0]]
counterfactual = exp.generate_counterfactuals(query_instance, total_CFs=2, desired_class="opposite")
counterfactual.visualize_as_dataframe()